# 🏥 Medalicus All-Systems Clinical Atlas — Google Colab Cloud Runner

This notebook runs the **Medalicus 3D Clinical Atlas** on Google Colab and creates a direct, 100% working public link with zero blank-screen issues.

In [ ]:
# STEP 1: Unpack Medalicus Bundle (Pre-built with 0 MB build errors)
import os, time, subprocess
from google.colab import files

# Clean any previous extract so we always run the latest build
!rm -rf /content/Medalicus /content/*.zip

print("Please upload the updated medalicus-colab-bundle.zip from your Downloads/Medalicus folder:")
uploaded = files.upload()
for fn in uploaded.keys():
    if fn.endswith(".zip"):
        !unzip -q -o "{fn}" -d /content/Medalicus
        break

if os.path.exists("/content/Medalicus"):
    %cd /content/Medalicus
    print("✓ Bundle unpacked successfully!")
else:
    print("Error: Could not find unpacked Medalicus directory.")


In [ ]:
# STEP 2: Start Clean Production Server on Port 3000 (No Host / CORS Blocks)
import subprocess, time

# Kill any old process on port 3000
!fuser -k 3000/tcp > /dev/null 2>&1

# Serve the pre-built optimized production bundle directly using Python's static server
# This bypasses Vite's host checking and WebSocket HMR blocking entirely!
proc = subprocess.Popen(['python3', '-m', 'http.server', '3000', '--directory', 'dist'])
time.sleep(2)
print('✓ Server active on port 3000!')

In [ ]:
# STEP 3: Generate Live Public HTTPS Links
# 1. Cloudflare Tunnel (100% public, works on any browser without cookie restrictions)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, re, time
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print('Generating secure public link...')
public_url = None
start = time.time()
while time.time() - start < 15:
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

# 2. Colab internal proxy URL as fallback
from google.colab.output import eval_js
colab_proxy = eval_js('google.colab.kernel.proxyPort(3000)')

print('\n' + '='*70)
if public_url:
    print('🎉 RECOMMENDED PUBLIC LINK (NO BLANK SCREEN):')
    print(f'👉 {public_url}')
    print('='*70)
print('\nAlternative Colab Internal Proxy Link:')
print(f'👉 {colab_proxy}')
print('='*70 + '\n')